# deckard Layers Walkthrough

This notebook explains the scripts in `deckard/layers`, with emphasis on `optimize.py` and the Hydra override path it depends on.

It is meant to be read alongside the more focused notebooks:

- [hydra.ipynb](hydra.ipynb) for composition and override syntax
- [optuna.ipynb](optuna.ipynb) for studies, trials, samplers, pruners, and storage
- [artifacts.ipynb](artifacts.ipynb) for end-to-end output and artifact handling
- [lifelines.ipynb](lifelines.ipynb) for survival-mode workflows
- [anjana.ipynb](anjana.ipynb) for a broader API tour across data, model, attack, and plotting layers

The goal here is narrower: show how the layer entrypoints fit together, how Hydra configuration flows into them, and where the optimize callback writes runtime files.

## How `deckard.layers.optimize` Works

`deckard.layers.optimize.DefaultOptimizerCallback` is the main bridge between Hydra and Optuna in Deckard.

At a high level, it:

- reads the composed Hydra config
- resolves per-run output paths such as `scores.json`, `params.yaml`, and log files
- creates or resumes the Optuna study from Hydra sweeper settings
- writes the final score payload back into the run directory

The most important idea is that the callback does not invent its own configuration shape. It consumes the same Hydra blocks used elsewhere in the notebook stack, so the behavior stays consistent with [hydra.ipynb](hydra.ipynb) and [optuna.ipynb](optuna.ipynb).

## Cross-Links and Nearby Surfaces

The layer scripts are easiest to understand as a small workflow graph:

- `optimize.py` handles Hydra callbacks, Optuna studies, and per-run file resolution.
- `plot.py` turns config-composed experiments into plots and can infer experiment settings from Hydra configuration.
- `progress_bar.py` reads Hydra and DVC metadata to estimate how many studies or trials are expected.

For worked examples, jump to:

- [hydra.ipynb](hydra.ipynb) for composition and override syntax
- [optuna.ipynb](optuna.ipynb) for study and storage behavior
- [artifacts.ipynb](artifacts.ipynb) for end-to-end model artifacts
- [lifelines.ipynb](lifelines.ipynb) for survival-mode configuration
- [anjana.ipynb](anjana.ipynb) for a broader layered API demonstration

## How `deckard.layers.plot` Works

`deckard.layers.plot.plot_main` is a dispatcher for two plotting workflows that share the same `plot_params_file` input.

It first resolves the Hydra config into a normalized plot block, then chooses a backend:

- [yellowbrick](../overview/extensions/index) when an experiment config is present, because the plot is built from fitted models
- [seaborn](../overview/extensions/index) when no experiment config is present and the input is tabular or aggregated results in a data file

That split matters because the inputs and outputs are different. Yellowbrick works from a live experiment configuration and writes plots directly from model objects. Seaborn works from a results table and can render either a single plot or a list of plots from a YAML spec. The same `plot_params_file` key is used in both cases; the presence of an experiment decides which backend interprets it.

The helper functions in this module exist mostly to make that dispatch reliable:

- `_resolve_plot_args_from_cfg` merges top-level Hydra keys with the nested [plot](../api/modules) block
- `_extract_backend` decides whether the request should be treated as Yellowbrick or Seaborn
- `_load_experiment_config` and `_load_yaml` bring file-based config into memory
- `_normalize_yellowbrick_plots` keeps the `plots` argument consistent across scalar and list forms

See [artifacts.ipynb](artifacts.ipynb) and [anjana.ipynb](anjana.ipynb) for examples of this layer in a broader workflow.

## How `deckard.layers.survival` Works

`deckard.layers.survival.survival_main` is the entrypoint for survival analysis workflows.

The structure is simpler than `plot.py`, but it still has two distinct modes:

- plot-only mode when the config contains explicit plot specifications
- experiment mode when the config describes a full survival experiment

The workflow starts by pulling a `survival` block out of the Hydra config, validating the raw `data` and [model](../api/modules) specs, and then normalizing the model name into a canonical survival fitter label. That normalization is what makes user-facing aliases like `coxphfitter` or `weibullaftfitter` collapse into the small set of supported modes.

If plot-only mode is active, the module instantiates the experiment config, loads the dataset into a dataframe, and hands the result to the lifelines plotting plugin. If not, it instantiates the experiment config and runs the full workflow.

For a worked example, see [lifelines.ipynb](lifelines.ipynb).